In [1]:
!python -m pip install medmnist
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install tensorflow

In [2]:
import os
import numpy as np

dataset_dir = r"D:\Projet ML\DATA\ChestMNIST"

npz_files = [f for f in os.listdir(dataset_dir) if f.endswith(".npz")]
npz_files.sort()

print("Fichiers trouvés :")
for f in npz_files:
    full_path = os.path.join(dataset_dir, f)
    size_gb = os.path.getsize(full_path) / (1024**3)
    print(f"- {f} | {size_gb:.2f} Go")

print("\n--- Inspection d'un fichier ---")
sample_file = os.path.join(dataset_dir, npz_files[0])
print("Fichier inspecté :", sample_file)

data = np.load(sample_file)

print("\nClés contenues dans le fichier :")
for key in data.files:
    arr = data[key]
    print(f"{key}: shape={arr.shape}, dtype={arr.dtype}")

Fichiers trouvés :
- chestmnist.npz | 0.08 Go
- chestmnist_128.npz | 1.33 Go
- chestmnist_224.npz | 3.62 Go
- chestmnist_64.npz | 0.37 Go

--- Inspection d'un fichier ---
Fichier inspecté : D:\Projet ML\DATA\ChestMNIST\chestmnist.npz

Clés contenues dans le fichier :
train_images: shape=(78468, 28, 28), dtype=uint8
val_images: shape=(11219, 28, 28), dtype=uint8
test_images: shape=(22433, 28, 28), dtype=uint8
train_labels: shape=(78468, 14), dtype=uint8
val_labels: shape=(11219, 14), dtype=uint8
test_labels: shape=(22433, 14), dtype=uint8


In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from medmnist import INFO
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Seed
seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)

# Chargement du dataset
dataset_path = r"D:\Projet ML\DATA\ChestMNIST\chestmnist_128.npz"
data = np.load(dataset_path)

X_train = data["train_images"]
X_val = data["val_images"]
X_test = data["test_images"]

y_train = data["train_labels"]
y_val = data["val_labels"]
y_test = data["test_labels"]

print("=== Shapes initiales ===")
print("X_train :", X_train.shape, X_train.dtype)
print("X_val   :", X_val.shape, X_val.dtype)
print("X_test  :", X_test.shape, X_test.dtype)
print("y_train :", y_train.shape, y_train.dtype)
print("y_val   :", y_val.shape, y_val.dtype)
print("y_test  :", y_test.shape, y_test.dtype)

# Ajout du canal
X_train = np.expand_dims(X_train, axis=-1)
X_val = np.expand_dims(X_val, axis=-1)
X_test = np.expand_dims(X_test, axis=-1)

# Normalisation
X_train = X_train.astype(np.float32) / 255.0
X_val = X_val.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0

# Labels en float32
y_train = y_train.astype(np.float32)
y_val = y_val.astype(np.float32)
y_test = y_test.astype(np.float32)

print("\n=== Shapes après préparation ===")
print("X_train :", X_train.shape, X_train.dtype)
print("X_val   :", X_val.shape, X_val.dtype)
print("X_test  :", X_test.shape, X_test.dtype)

# Labels officiels
info = INFO["chestmnist"]

if isinstance(info["label"], dict):
    label_names = [info["label"][str(i)] if str(i) in info["label"] else info["label"][i] for i in range(len(info["label"]))]
else:
    label_names = list(info["label"])

print("\n=== Labels ===")
for i, name in enumerate(label_names):
    print(i, ":", name)

# Comptage du nombre de labels positifs par image
train_positive_counts = y_train.sum(axis=1)
val_positive_counts = y_val.sum(axis=1)
test_positive_counts = y_test.sum(axis=1)

print("\n=== Répartition images normales / anormales ===")
print("Train normales  :", np.sum(train_positive_counts == 0))
print("Train anormales :", np.sum(train_positive_counts > 0))
print("Val normales    :", np.sum(val_positive_counts == 0))
print("Val anormales   :", np.sum(val_positive_counts > 0))
print("Test normales   :", np.sum(test_positive_counts == 0))
print("Test anormales  :", np.sum(test_positive_counts > 0))

# Option AE : entraînement sur images sans label positif uniquement
X_train_normal = X_train[train_positive_counts == 0]
X_val_normal = X_val[val_positive_counts == 0]

print("\n=== Données utilisées pour entraîner l'AE ===")
print("X_train_normal :", X_train_normal.shape)
print("X_val_normal   :", X_val_normal.shape)

# Datasets TensorFlow AE
batch_size = 32

train_ds_ae = tf.data.Dataset.from_tensor_slices((X_train_normal, X_train_normal))
train_ds_ae = train_ds_ae.shuffle(buffer_size=len(X_train_normal), seed=seed, reshuffle_each_iteration=True)
train_ds_ae = train_ds_ae.batch(batch_size).prefetch(tf.data.AUTOTUNE)

val_ds_ae = tf.data.Dataset.from_tensor_slices((X_val_normal, X_val_normal))
val_ds_ae = val_ds_ae.batch(batch_size).prefetch(tf.data.AUTOTUNE)

test_ds_ae = tf.data.Dataset.from_tensor_slices((X_test, X_test))
test_ds_ae = test_ds_ae.batch(batch_size).prefetch(tf.data.AUTOTUNE)

print("\n=== Vérification datasets AE ===")
print("Train AE :", train_ds_ae)
print("Val AE   :", val_ds_ae)
print("Test AE  :", test_ds_ae)

for batch_x, batch_y in train_ds_ae.take(1):
    print("\n=== Batch AE ===")
    print("Input shape  :", batch_x.shape)
    print("Target shape :", batch_y.shape)
    print("Min pixel    :", tf.reduce_min(batch_x).numpy())
    print("Max pixel    :", tf.reduce_max(batch_x).numpy())

# Autoencodeur convolutionnel
encoder_inputs = layers.Input(shape=(128, 128, 1))

x = layers.Conv2D(32, 3, activation="relu", padding="same")(encoder_inputs)
x = layers.MaxPooling2D(2, padding="same")(x)

x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
x = layers.MaxPooling2D(2, padding="same")(x)

x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
encoded = layers.MaxPooling2D(2, padding="same", name="latent_space")(x)

x = layers.Conv2D(128, 3, activation="relu", padding="same")(encoded)
x = layers.UpSampling2D(2)(x)

x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
x = layers.UpSampling2D(2)(x)

x = layers.Conv2D(32, 3, activation="relu", padding="same")(x)
x = layers.UpSampling2D(2)(x)

decoded = layers.Conv2D(1, 3, activation="sigmoid", padding="same")(x)

ae_model = models.Model(encoder_inputs, decoded, name="conv_autoencoder_chestmnist")

ae_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse"
)

print("\n=== Summary de l'Autoencodeur ===")
ae_model.summary()

callbacks_ae = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        "best_conv_autoencoder.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

=== Shapes initiales ===
X_train : (78468, 128, 128) uint8
X_val   : (11219, 128, 128) uint8
X_test  : (22433, 128, 128) uint8
y_train : (78468, 14) uint8
y_val   : (11219, 14) uint8
y_test  : (22433, 14) uint8

=== Shapes après préparation ===
X_train : (78468, 128, 128, 1) float32
X_val   : (11219, 128, 128, 1) float32
X_test  : (22433, 128, 128, 1) float32

=== Labels ===
0 : atelectasis
1 : cardiomegaly
2 : effusion
3 : infiltration
4 : mass
5 : nodule
6 : pneumonia
7 : pneumothorax
8 : consolidation
9 : edema
10 : emphysema
11 : fibrosis
12 : pleural
13 : hernia

=== Répartition images normales / anormales ===
Train normales  : 42405
Train anormales : 36063
Val normales    : 6079
Val anormales   : 5140
Test normales   : 11928
Test anormales  : 10505

=== Données utilisées pour entraîner l'AE ===
X_train_normal : (42405, 128, 128, 1)
X_val_normal   : (6079, 128, 128, 1)

=== Vérification datasets AE ===
Train AE : <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 128, 128, 1)

Model: "conv_autoencoder_chestmnist"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 128, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ latent_space (MaxPooling2D)     │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d (UpSampling2D)    │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 32, 32, 64)     │        73,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_1 (UpSampling2D)  │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 64, 64, 32)     │        18,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_2 (UpSampling2D)  │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 128, 128, 1)    │           289 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 332,801 (1.27 MB)

 Trainable params: 332,801 (1.27 MB)

 Non-trainable params: 0 (0.00 B)

## Initialisation de la composante de détection d’anomalies par autoencodeur

La section de détection d’anomalies est initialisée à partir du dataset **ChestMNIST_128** déjà utilisé dans la partie supervisée. Les images ont été préparées de la même manière :

- ajout d’un canal explicite ;
- normalisation dans l’intervalle `[0,1]` ;
- conservation des labels pour distinguer les cas normaux et anormaux.

### 1. Principe retenu

L’approche choisie ici consiste à entraîner un **autoencodeur convolutionnel (AE)** sur les images considérées comme **normales**, c’est-à-dire les radiographies ne possédant aucun label positif. L’idée est que le modèle apprenne à bien reconstruire les cas normaux, tandis que les images atypiques ou pathologiques devraient produire une erreur de reconstruction plus élevée.

Cette stratégie est cohérente avec l’objectif du sujet : obtenir un **score d’anomalie fondé sur la reconstruction** afin d’identifier des cas atypiques, ambigus ou hors distribution.

### 2. Architecture de l’AE

L’autoencodeur construit est un **autoencodeur convolutionnel symétrique** :

- partie encodeur :
  - convolutions successives
  - réductions spatiales par max pooling
- espace latent :
  - représentation compressée en **16 × 16 × 128**
- partie décodeur :
  - convolutions
  - remontée de résolution par `UpSampling2D`
  - sortie finale en **sigmoïde**

Le modèle totalise environ **332 801 paramètres**, ce qui reste relativement léger par rapport aux architectures supervisées testées précédemment.

### 3. Lecture pratique

L’AE est plus compact que les modèles de classification, ce qui est intéressant dans un cadre expérimental où l’on souhaite tester rapidement un score d’anomalie basé sur l’erreur de reconstruction. La prochaine étape consiste à l’entraîner sur les images normales, puis à comparer les erreurs de reconstruction obtenues sur :

- les images normales ;
- les images pathologiques ;
- quelques cas individuels représentatifs.

In [ ]:
history_ae = ae_model.fit(
    train_ds_ae,
    validation_data=val_ds_ae,
    epochs=15,
    callbacks=callbacks_ae,
    verbose=1
)

Epoch 1/15
 905/1326 ━━━━━━━━━━━━━━━━━━━━ 44s 106ms/step - loss: 0.0078